# Sweep Runner

Select an experiment, check W&B for completed runs, and launch remaining hyperparameter combinations.

In [1]:
import os
import glob
import ipywidgets as widgets
import itertools
import subprocess
from pathlib import Path

import wandb
import yaml
from omegaconf import OmegaConf

REPO_ROOT = Path(os.path.abspath("")).parent
CONF_DIR = REPO_ROOT / "JacobianODE" / "jacobians" / "conf"
EXP_DIR = CONF_DIR / "experiment"

## 1. Select an experiment

In [2]:
# Discover experiment YAML files (exclude old/ subdirectory)
experiment_files = sorted(
    p for p in EXP_DIR.glob("*.yaml")
    if p.is_file()
)

experiments = {p.stem: p for p in experiment_files}
for i, name in enumerate(experiments):
    print(f"  [{i}] {name}")

dropdown = widgets.Dropdown(
    options=list(experiments.keys()),
    description="Experiment:",
    value=list(experiments.keys())[0]
)
display(dropdown)

# Then read dropdown.value whenever you need it

  [0] lorenz_partial_spline_coupling_geometric_noise_sweep
  [1] lorenz_spline_coupling_geometric_noise_sweep
  [2] wmtask_spline_coupling_geometric_noise_sweep
  [3] wmtask_spline_coupling_sweep


Dropdown(description='Experiment:', options=('lorenz_partial_spline_coupling_geometric_noise_sweep', 'lorenz_s…

In [3]:
# ── Pick one ──────────────────────────────────────────────────────────────────
# EXPERIMENT = list(experiments.keys())[0]  # <-- change index or set name directly
EXPERIMENT = dropdown.value

print(f"Selected: {EXPERIMENT}")

Selected: lorenz_partial_spline_coupling_geometric_noise_sweep


## 2. Parse experiment config (sweep grid, W&B project/group)

In [4]:
# Load experiment YAML and the base configs it depends on to resolve interpolations
exp_path = experiments[EXPERIMENT]
exp_cfg = OmegaConf.load(exp_path)

# Load base configs for variable resolution
data_name = None
for d in OmegaConf.to_container(exp_cfg.get("defaults", []), resolve=False) or []:
    if isinstance(d, dict) and "override /data" in d:
        data_name = d["override /data"]
if data_name:
    data_cfg = OmegaConf.load(CONF_DIR / "data" / f"{data_name}.yaml")
else:
    data_cfg = OmegaConf.create()

model_name = None
for d in OmegaConf.to_container(exp_cfg.get("defaults", []), resolve=False) or []:
    if isinstance(d, dict) and "override /model" in d:
        model_name = d["override /model"]
if model_name:
    model_cfg = OmegaConf.load(CONF_DIR / "model" / f"{model_name}.yaml")
else:
    model_cfg = OmegaConf.create()

training_cfg = OmegaConf.load(CONF_DIR / "training" / "training.yaml")

# Merge: base < data < model < training < experiment (experiment wins)
# Exclude the hydra block — it contains ${now:...} which is a Hydra-only
# resolver and will fail under plain OmegaConf.resolve().
exp_cfg_no_hydra = {k: v for k, v in OmegaConf.to_container(exp_cfg).items()
                    if k not in ("defaults", "hydra")}

base_cfg = OmegaConf.load(CONF_DIR / "config.yaml")
# Also strip hydra from base config
base_no_hydra = {k: v for k, v in OmegaConf.to_container(base_cfg).items()
                 if k != "hydra"}

merged = OmegaConf.merge(
    base_no_hydra,
    {"data": data_cfg},
    {"model": model_cfg},
    {"training": training_cfg},
    exp_cfg_no_hydra,
)
OmegaConf.resolve(merged)

# ── Extract W&B coordinates ──────────────────────────────────────────────────
WANDB_ENTITY = merged.get("wandb_entity", "JacobianODE")
WANDB_PROJECT = merged.get("wandb_project", None)
WANDB_GROUP = merged.get("wandb_group", None)

# ── Extract sweep grid ───────────────────────────────────────────────────────
# Read directly from the raw experiment YAML (hydra block), no resolution needed
sweep_params_raw = OmegaConf.to_container(
    exp_cfg.hydra.sweeper.params, resolve=False
)

# Parse each param: "0,1e-6,..." -> list of float values
sweep_params = {}
for key, val_str in sweep_params_raw.items():
    values = [float(v) for v in val_str.split(",")]
    sweep_params[key] = values

# Full grid = cartesian product
param_names = list(sweep_params.keys())
param_values = list(sweep_params.values())
full_grid = [dict(zip(param_names, combo)) for combo in itertools.product(*param_values)]

print(f"W&B entity:  {WANDB_ENTITY}")
print(f"W&B project: {WANDB_PROJECT}")
print(f"W&B group:   {WANDB_GROUP}")
print(f"Sweep params: {param_names}")
print(f"Grid sizes:   {[len(v) for v in param_values]}")
print(f"Total combinations: {len(full_grid)}")

W&B entity:  JacobianODE
W&B project: Lorenz_INDall_N25_D1_NormTrue_T3__JacobianODE
W&B group:   spline_coupling__geometric_noise__sweep_lc_x_kl_dyn
Sweep params: ['training.lightning.loop_closure_weight', 'training.lightning.kl_dyn_weight']
Grid sizes:   [9, 9]
Total combinations: 81


## 3. Query W&B for completed runs

In [5]:
import math
from datetime import datetime, timezone

def _get_config_value(run_config, dotted_key):
    """Extract a value from a W&B run config using a dotted key like
    'training.lightning.loop_closure_weight'. Handles both nested dicts
    and flat dotted keys."""
    # Try nested dict traversal first
    parts = dotted_key.split(".")
    obj = run_config
    for part in parts:
        if isinstance(obj, dict) and part in obj:
            obj = obj[part]
        else:
            obj = None
            break
    if obj is not None:
        return obj
    # Fallback: flat key
    return run_config.get(dotted_key)


def _to_float(val):
    """Safely coerce a value to float, returning NaN on failure."""
    try:
        return float(val)
    except (TypeError, ValueError):
        return float("nan")


def meets_early_stopping_criterion(run, monitor="mean val loss", patience=5,
                                    percent_thresh=0.01, min_epochs=10):
    """Check if a run's loss history shows it had converged before it ended.
    
    This catches runs that were killed (SLURM timeout, crash) AFTER the loss
    had already plateaued — i.e., they would have early-stopped if given more
    time. Also used to confirm finished runs that early-stopped.
    
    Returns (met_criterion: bool, last_epoch: int or None).
    """
    try:
        history = run.history(keys=[monitor, "epoch"], pandas=True)
    except Exception:
        return False, None

    if history.empty or monitor not in history.columns:
        return False, None

    # Drop NaN rows for the monitor metric (validation is logged less often)
    history = history.dropna(subset=[monitor])
    if len(history) < 2:
        return False, None

    losses = [_to_float(v) for v in history[monitor].tolist()]
    epochs = [_to_float(v) for v in history["epoch"].tolist()] if "epoch" in history.columns else list(range(len(losses)))
    last_epoch = int(epochs[-1]) if epochs else None

    # Don't consider convergence before min_epochs
    start_idx = 0
    for i, e in enumerate(epochs):
        if e >= min_epochs:
            start_idx = i
            break
    else:
        # Never reached min_epochs
        return False, last_epoch

    # Replay the PercentEarlyStopping logic on the loss history
    wait_count = 0
    prev_loss = losses[start_idx]
    for i in range(start_idx + 1, len(losses)):
        current = losses[i]
        if not math.isfinite(current):
            wait_count += 1
            if wait_count >= patience:
                return True, last_epoch
            continue
        if not math.isfinite(prev_loss):
            prev_loss = current
            wait_count = 0
            continue

        if prev_loss > current:
            pct_improvement = (prev_loss - current) / prev_loss
            if pct_improvement < percent_thresh:
                wait_count += 1
            else:
                wait_count = 0
        else:
            wait_count += 1

        prev_loss = current
        if wait_count >= patience:
            return True, last_epoch

    return False, last_epoch


def hit_slurm_walltime(run, timeout_min, tolerance_min=5):
    """Check if a crashed/failed run's wall-clock duration is within
    `tolerance_min` minutes of the SLURM timeout, indicating it was
    killed by the job scheduler rather than a real error."""
    try:
        created = datetime.fromisoformat(run.created_at.replace("Z", "+00:00"))
        # W&B heartbeat_at is the last time the run was alive
        heartbeat = run.heartbeat_at or run.updated_at
        ended = datetime.fromisoformat(heartbeat.replace("Z", "+00:00"))
        duration_min = (ended - created).total_seconds() / 60.0
        return duration_min >= (timeout_min - tolerance_min)
    except Exception:
        return False

In [6]:
api = wandb.Api(timeout=90)
project_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

run_filters = {}
if WANDB_GROUP:
    run_filters["group"] = WANDB_GROUP

try:
    all_runs = api.runs(project_path, filters=run_filters if run_filters else None)
    print(f"Found {len(all_runs)} total runs in {project_path}")
    if WANDB_GROUP:
        print(f"  (filtered to group: {WANDB_GROUP})")
except Exception as e:
    all_runs = []
    print(f"Could not fetch runs: {e}")
    print("(Project may not exist yet — all combinations will be marked as remaining.)")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/eisenaj/.netrc.


Found 58 total runs in JacobianODE/Lorenz_INDall_N25_D1_NormTrue_T3__JacobianODE
  (filtered to group: spline_coupling__geometric_noise__sweep_lc_x_kl_dyn)


In [7]:
# ── Classify runs ─────────────────────────────────────────────────────────────
max_epochs = int(OmegaConf.select(merged, "training.trainer_params.max_epochs", default=1000))
es_patience = int(OmegaConf.select(merged, "training.early_stopping.early_stopping_patience", default=5))
es_pct_thresh = float(OmegaConf.select(merged, "training.early_stopping.percent_thresh", default=0.01))
es_min_epochs = int(OmegaConf.select(merged, "training.early_stopping.min_epochs", default=10))
es_monitor = OmegaConf.select(merged, "training.early_stopping.monitor", default="mean val loss")

# SLURM walltime from the slurm config used by this experiment
slurm_cfg_name = None
for d in OmegaConf.to_container(exp_cfg.get("defaults", []), resolve=False) or []:
    if isinstance(d, dict) and "override /slurm" in d:
        slurm_cfg_name = d["override /slurm"]
if slurm_cfg_name is None:
    # Fall back to base config default
    slurm_cfg_name = OmegaConf.select(base_cfg, "defaults", default=[])
    # base config defaults list: find slurm entry
    for d in OmegaConf.to_container(OmegaConf.load(CONF_DIR / "config.yaml").get("defaults", []), resolve=False) or []:
        if isinstance(d, dict) and "slurm" in d:
            slurm_cfg_name = d["slurm"]
            break
        elif isinstance(d, str) and d.startswith("slurm:"):
            slurm_cfg_name = d.split(":")[1].strip()
            break

slurm_timeout_min = None
if slurm_cfg_name and slurm_cfg_name != "none":
    slurm_cfg_path = CONF_DIR / "slurm" / f"{slurm_cfg_name}.yaml"
    if slurm_cfg_path.exists():
        slurm_cfg = OmegaConf.load(slurm_cfg_path)
        slurm_timeout_min = OmegaConf.select(slurm_cfg, "hydra.launcher.timeout_min", default=None)

WALLTIME_TOLERANCE_MIN = 5
print(f"SLURM timeout: {slurm_timeout_min} min" + (f"  (tolerance: {WALLTIME_TOLERANCE_MIN} min)" if slurm_timeout_min else " (not set)"))

finished_configs = []   # list of param dicts that are done
running_configs = []    # currently running
failed_configs = []     # crashed / failed and NOT converged

n_finished = 0
n_early_stopped = 0
n_running = 0
n_failed = 0
n_converged_crashed = 0
n_walltime_killed = 0

for run in all_runs:
    # Extract the swept param values from this run's config
    run_params = {}
    skip = False
    for pname in param_names:
        val = _get_config_value(run.config, pname)
        if val is None:
            skip = True
            break
        run_params[pname] = float(val)
    if skip:
        continue

    if run.state == "finished":
        finished_configs.append(run_params)
        n_finished += 1
        # Check if it early-stopped (informational)
        last_epoch = run.summary.get("epoch", run.summary.get("trainer/current_epoch"))
        if last_epoch is not None and int(last_epoch) < max_epochs - 1:
            n_early_stopped += 1

    elif run.state == "running":
        running_configs.append(run_params)
        n_running += 1

    elif run.state in ("crashed", "failed"):
        parts = [f"{k.split('.')[-1]}={run_params[k]}" for k in param_names]
        reason = None

        # Check 1: did the loss converge before the crash?
        converged, last_ep = meets_early_stopping_criterion(
            run,
            monitor=es_monitor,
            patience=es_patience,
            percent_thresh=es_pct_thresh,
            min_epochs=es_min_epochs,
        )
        if converged:
            reason = f"converged (epoch {last_ep})"
            n_converged_crashed += 1

        # Check 2: did it run until the SLURM walltime?
        if reason is None and slurm_timeout_min is not None:
            if hit_slurm_walltime(run, slurm_timeout_min, tolerance_min=WALLTIME_TOLERANCE_MIN):
                reason = "hit SLURM walltime"
                n_walltime_killed += 1

        if reason is not None:
            finished_configs.append(run_params)  # count as done
            print(f"  {run.id}: {reason} — {', '.join(parts)}")
        else:
            failed_configs.append(run_params)
            n_failed += 1

print(f"\nFinished:              {n_finished}  ({n_early_stopped} early-stopped)")
print(f"Crashed but converged: {n_converged_crashed}  (counted as done)")
print(f"Hit SLURM walltime:    {n_walltime_killed}  (counted as done)")
print(f"Running:               {n_running}")
print(f"Crashed/Failed:        {n_failed}  (need re-run)")

SLURM timeout: 180 min  (tolerance: 5 min)
  6atiiue8: hit SLURM walltime — loop_closure_weight=0.001, kl_dyn_weight=0.001
  o5zs5hp7: hit SLURM walltime — loop_closure_weight=0.001, kl_dyn_weight=0.01
  sa6txyhh: hit SLURM walltime — loop_closure_weight=0.001, kl_dyn_weight=0.1
  p29oyx3a: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=0.0
  q6jykzmg: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=0.0001
  z74s2zhq: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=1e-05
  09at3h5i: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=1e-06
  4rl9mvew: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=0.01
  k9shz73q: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=0.1
  zb1yekmv: hit SLURM walltime — loop_closure_weight=0.01, kl_dyn_weight=0.001
  s7c542og: converged (epoch 58) — loop_closure_weight=0.01, kl_dyn_weight=1.0
  ajw1ps7j: hit SLURM walltime — loop_closure_weight=0.1, kl_dyn_weight=0.0
  k5ap3eks: hit 

## 4. Compute remaining hyperparameter combinations

In [8]:
def params_match(a, b, tol=1e-12):
    """Check if two param dicts match (float comparison with tolerance)."""
    for key in a:
        va, vb = float(a[key]), float(b[key])
        if va == 0 and vb == 0:
            continue
        if va == 0 or vb == 0:
            return False
        if abs(va - vb) / max(abs(va), abs(vb)) > tol:
            return False
    return True


def is_done_or_running(combo, done_list, running_list):
    """Check if a grid combo is already finished or running."""
    for done in done_list:
        if params_match(combo, done):
            return True
    for running in running_list:
        if params_match(combo, running):
            return True
    return False


# ── INCLUDE_FAILED: set True to re-run failed jobs, False to skip them ────────
INCLUDE_FAILED = True

remaining = []
for combo in full_grid:
    if is_done_or_running(combo, finished_configs, running_configs):
        continue
    if not INCLUDE_FAILED and any(params_match(combo, f) for f in failed_configs):
        continue
    remaining.append(combo)

print(f"Total grid:     {len(full_grid)}")
print(f"Already done:   {len(finished_configs)}")
print(f"Running:        {len(running_configs)}")
print(f"Remaining:      {len(remaining)}")

if remaining:
    print(f"\nFirst 5 remaining combos:")
    for combo in remaining[:5]:
        parts = [f"{k.split('.')[-1]}={v}" for k, v in combo.items()]
        print(f"  {', '.join(parts)}")
    if len(remaining) > 5:
        print(f"  ... and {len(remaining) - 5} more")

Total grid:     81
Already done:   57
Running:        0
Remaining:      24

First 5 remaining combos:
  loop_closure_weight=0.001, kl_dyn_weight=1.0
  loop_closure_weight=0.1, kl_dyn_weight=0.001
  loop_closure_weight=0.1, kl_dyn_weight=0.01
  loop_closure_weight=0.1, kl_dyn_weight=0.1
  loop_closure_weight=0.1, kl_dyn_weight=1.0
  ... and 19 more


## 5. Generate bash command

In [11]:
# ── OPTIONS ────────────────────────────────────────────────────────────────────
RUN_MODE = "remaining"  # "full" = run entire experiment, "remaining" = only missing combos
SLURM = True            # True = submit via SLURM (slurm=default), False = local (slurm=none)
DRY_RUN = True          # True = print command only, False = also execute it

ENV_PREFIX = "HYDRA_FULL_ERROR=1"


def _fmt_value(v):
    """Format a float for Hydra override (avoid scientific notation issues)."""
    if v == 0:
        return "0"
    if v == int(v) and abs(v) < 1e6:
        return str(int(v))
    return repr(v)


def _make_temp_experiment(exp_path, exp_dir):
    """Create a temporary copy of the experiment YAML with the hydra block
    removed, so it can be used with a different sweeper without schema conflicts.
    Returns the experiment name (stem) to use in the override."""
    import shutil
    raw = yaml.safe_load(exp_path.read_text())
    raw.pop("hydra", None)
    temp_name = f"_tmp_{exp_path.stem}"
    temp_path = exp_dir / f"{temp_name}.yaml"
    with open(temp_path, "w") as f:
        yaml.dump(raw, f, default_flow_style=False, sort_keys=False)
    return temp_name, temp_path


if RUN_MODE == "full":
    cmd = (
        f"cd {REPO_ROOT} && {ENV_PREFIX} "
        f"python -m JacobianODE.jacobians.run_jacobians --multirun "
        f"experiment={EXPERIMENT}"
    )
    if not SLURM:
        cmd += " slurm=none"
    print("Command (full sweep):")
    print(cmd)

elif RUN_MODE == "remaining":
    if not remaining:
        print("Nothing remaining to run!")
    else:
        # Check if remaining is a clean sub-grid (cartesian product)
        remaining_value_sets = {pname: set() for pname in param_names}
        for combo in remaining:
            for pname in param_names:
                remaining_value_sets[pname].add(combo[pname])
        remaining_as_subgrid = list(
            itertools.product(*[sorted(remaining_value_sets[p]) for p in param_names])
        )
        is_subgrid = len(remaining_as_subgrid) == len(remaining)

        if is_subgrid and len(remaining) > 1:
            # Clean sub-grid: use default Hydra sweeper with comma-separated values
            overrides = []
            for pname in param_names:
                vals = sorted(remaining_value_sets[pname])
                val_str = ",".join(_fmt_value(v) for v in vals)
                overrides.append(f"'{pname}={val_str}'")

            cmd = (
                f"cd {REPO_ROOT} && {ENV_PREFIX} "
                f"python -m JacobianODE.jacobians.run_jacobians --multirun "
                f"experiment={EXPERIMENT} "
                + " ".join(overrides)
            )
            if not SLURM:
                cmd += " slurm=none"
            print(f"Command (sub-grid, {len(remaining)} combos):")
            print(cmd)
        else:
            # Not a clean sub-grid: use hydra-list-sweeper.
            # The experiment YAML has hydra.sweeper.params baked in, which
            # conflicts with the list sweeper schema (applied before CLI
            # overrides). Workaround: write a temp experiment config without
            # the hydra block.
            temp_name, temp_path = _make_temp_experiment(exp_path, EXP_DIR)

            overrides = ["'hydra/sweeper=list'"]
            for pname in param_names:
                vals = [_fmt_value(combo[pname]) for combo in remaining]
                val_str = ",".join(vals)
                overrides.append(f"'+hydra.sweeper.list_params.{pname}=[{val_str}]'")

            cmd = (
                f"cd {REPO_ROOT} && {ENV_PREFIX} "
                f"python -m JacobianODE.jacobians.run_jacobians --multirun "
                f"experiment={temp_name} "
                + " ".join(overrides)
            )
            if not SLURM:
                cmd += " slurm=none"
            print(f"Command (list sweep, {len(remaining)} combos):")
            print(f"Temp experiment config: {temp_path}")
            print(cmd)

Command (list sweep, 24 combos):
cd /orcd/home/002/eisenaj/code/JacobianODE && HYDRA_FULL_ERROR=1 python -m JacobianODE.jacobians.run_jacobians --multirun experiment=lorenz_partial_spline_coupling_geometric_noise_sweep 'hydra/sweeper=list' '~hydra.sweeper.params' '+hydra.sweeper.list_params.training.lightning.loop_closure_weight=[0.001,0.1,0.1,0.1,0.1,0.1,1,1,1,1,1,1,1,1,1,10,10,10,10,10,10,10,10,10]' '+hydra.sweeper.list_params.training.lightning.kl_dyn_weight=[1,0.001,0.01,0.1,1,10,0,1e-06,1e-05,0.0001,0.001,0.01,0.1,1,10,0,1e-06,1e-05,0.0001,0.001,0.01,0.1,1,10]'


In [12]:
# ── Execute (only if DRY_RUN is False) ────────────────────────────────────────
if not DRY_RUN and RUN_MODE == "remaining" and not remaining:
    print("Nothing to run.")
elif not DRY_RUN:
    print(f"Launching...\n")
    result = subprocess.run(cmd, shell=True, capture_output=False)
    print(f"\nExit code: {result.returncode}")
    # Clean up temp experiment config if one was created
    if "temp_path" in dir() and temp_path.exists():
        temp_path.unlink()
        print(f"Cleaned up {temp_path}")
else:
    print("\n(DRY_RUN=True — set to False and re-run this cell to execute)")

Launching...



Traceback (most recent call last):
  File "/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/hydra/_internal/config_loader_impl.py", line 542, in _compose_config_from_defaults_list
    cfg.merge_with(loaded.config)
  File "/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/omegaconf/basecontainer.py", line 492, in merge_with
    self._format_and_raise(key=None, value=None, cause=e)
  File "/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/omegaconf/base.py", line 231, in _format_and_raise
    format_and_raise(
  File "/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/omegaconf/_utils.py", line 819, in format_and_raise
    _raise(ex, cause)
  File "/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/omegaconf/_utils.py", line 797, in _raise
    raise ex.with_traceback(sys.exc_info()[2])  # set env var OC_CAUSE=1 for full trace
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/eisenaj/code/J


Exit code: 1
